In [ ]:
import sagemaker
import boto3
import os
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

In [ ]:
session = sagemaker.Session()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()

bucket = session.default_bucket()
prefix = "tarea-6-processing-byoc"

print("Region:", region)
print("Role:", role)
print("Bucket:", bucket)
print("Prefix:", prefix)

In [ ]:
project_root = os.path.abspath(".")

local_data_path = os.path.join(project_root, "data", "raw", "sales_train.csv")
script_path = os.path.join(project_root, "processing", "prep.py")
dockerfile_dir = os.path.join(project_root, "processing", "container")

print(local_data_path)
print(script_path)
print(dockerfile_dir)

In [ ]:
s3_input_data = session.upload_data(
    path=local_data_path,
    bucket=bucket,
    key_prefix=f"{prefix}/input"
)

print("S3 input path:", s3_input_data)

In [ ]:
import subprocess

account_id = boto3.client("sts").get_caller_identity()["Account"]
repository_name = "tarea-6-processing-byoc"
image_tag = "latest"

image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repository_name}:{image_tag}"

print("Account ID:", account_id)
print("Image URI:", image_uri)

In [ ]:
ecr = boto3.client("ecr")

try:
    ecr.describe_repositories(repositoryNames=[repository_name])
    print(f"El repositorio {repository_name} ya existe.")
except ecr.exceptions.RepositoryNotFoundException:
    ecr.create_repository(repositoryName=repository_name)
    print(f"Repositorio {repository_name} creado.")

In [ ]:
login_password = boto3.client("ecr").get_authorization_token()["authorizationData"][0]["authorizationToken"]

print("Listo para hacer login a ECR desde terminal o notebook.")

In [ ]:
s3_output_path = f"s3://{bucket}/{prefix}/output/"
print("S3 output path:", s3_output_path)

In [ ]:
script_processor = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=session
)

In [ ]:
script_processor.run(
    code=script_path,
    inputs=[
        ProcessingInput(
            source=s3_input_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=s3_output_path
        )
    ]
)